# Fine-tune NER — NlpHUST/ner-vietnamese-electra-base

Notebook fine-tune mô hình **`NlpHUST/ner-vietnamese-electra-base`** trên bộ dữ liệu pháp lý (BIO tagging) với các kỹ thuật tối ưu:

- **Subword label alignment** (chỉ gán label cho first subtoken, còn lại `-100`)
- **Mixed precision (fp16)** + **gradient accumulation**
- **Layer-wise Learning Rate Decay (LLRD)** — kỹ thuật mạnh cho fine-tune transformer
- **Linear warmup + decay**, **weight decay**, **gradient clipping**
- **Label smoothing**
- **Early stopping** theo `eval_f1` (seqeval entity-level)
- **Dynamic padding** với `DataCollatorForTokenClassification`
- **Reproducible seed**, auto-discover label space, lưu `id2label` cùng model

Input:
- `dataset/ner_finetune_data.jsonl` (train)
- `dataset/ner_val.jsonl` (validation)
- `dataset/ner_test.jsonl` (test)

Output: `models/ner-legal-vi-electra/` (model + tokenizer + label mapping).

## 1. Cài đặt môi trường

In [1]:
# Chạy 1 lần nếu chưa cài. Bỏ comment nếu cần.
%pip install -q "transformers>=4.41" "datasets>=2.19" "accelerate>=0.30" "seqeval" "evaluate" "torch" "numpy" "pandas"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00


In [2]:
import os
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

torch 2.10.0+cu128 | cuda True
device: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Cấu hình

In [3]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Paths ---
PROJECT_ROOT = Path("..").resolve()
DATA_DIR  = PROJECT_ROOT / "/content/dataset"
TRAIN_FILE = DATA_DIR / "ner_finetune_data.jsonl"
VAL_FILE   = DATA_DIR / "ner_val.jsonl"
TEST_FILE  = DATA_DIR / "ner_test.jsonl"

OUTPUT_DIR = PROJECT_ROOT / "models" / "ner-legal-vi-electra"
LOG_DIR    = PROJECT_ROOT / "models" / "runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# --- Model ---
BASE_MODEL = "NlpHUST/ner-vietnamese-electra-base"
MAX_LENGTH = 256  # câu trong dataset rất ngắn

# --- Training hyperparams (tối ưu cho GPU 96GB — H100 / A100-80G+) ---
NUM_EPOCHS        = 30       # trần; early stopping sẽ dừng sớm
BATCH_SIZE_TRAIN  = 64       # effective batch = 64 — sweet spot cho NER fine-tune
BATCH_SIZE_EVAL   = 128
GRAD_ACCUM        = 1        # không cần với GPU lớn
BASE_LR           = 3e-5     # backbone Electra
CLASSIFIER_LR     = 1e-4     # head reinit — LR cao hơn
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.1
LABEL_SMOOTHING   = 0.1
MAX_GRAD_NORM     = 1.0
LLRD_DECAY        = 0.9      # layer-wise LR decay (1.0 = tắt)
EARLY_STOP_PAT    = 5        # kiên nhẫn hơn vì tổng epoch lớn hơn

# --- Precision ---
# Ưu tiên bf16 (ổn định, không cần loss scaler) trên Ampere/Hopper.
HAS_CUDA = torch.cuda.is_available()
SUPPORTS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
USE_BF16 = SUPPORTS_BF16
USE_FP16 = HAS_CUDA and not SUPPORTS_BF16

# Bật TF32 cho matmul / cuDNN — tăng tốc ~1.5x không ảnh hưởng chất lượng.
if HAS_CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print("Train:", TRAIN_FILE)
print("Val  :", VAL_FILE)
print("Test :", TEST_FILE)
print("Out  :", OUTPUT_DIR)
print(f"precision: bf16={USE_BF16}  fp16={USE_FP16}  tf32=on")

Train: /content/dataset/ner_finetune_data.jsonl
Val  : /content/dataset/ner_val.jsonl
Test : /content/dataset/ner_test.jsonl
Out  : /models/ner-legal-vi-electra
precision: bf16=True  fp16=False  tf32=on


## 3. Nạp dữ liệu JSONL → HuggingFace Dataset

In [4]:
def load_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            # kiểm tra nhất quán
            assert len(obj["tokens"]) == len(obj["labels"]), f"Mismatch at id={obj.get('id')}"
            rows.append({
                "id": obj.get("id"),
                "tokens": obj["tokens"],
                "ner_tags": obj["labels"],  # tạm giữ tên chuỗi, sẽ map sang int
            })
    return rows

train_rows = load_jsonl(TRAIN_FILE)
val_rows   = load_jsonl(VAL_FILE)
test_rows  = load_jsonl(TEST_FILE)

print(f"train={len(train_rows)}  val={len(val_rows)}  test={len(test_rows)}")
print("ví dụ:", train_rows[0])

train=6403  val=1279  test=1288
ví dụ: {'id': 0, 'tokens': ['Khoản', '1', 'Điều', '46', 'quy', 'định', 'về', 'bảo', 'vệ', 'dữ', 'liệu', 'cá', 'nhân', 'áp', 'dụng', 'trong', 'lĩnh', 'vực', 'số', '.'], 'ner_tags': ['B-CLAUSE', 'I-CLAUSE', 'B-ARTICLE', 'I-ARTICLE', 'O', 'O', 'O', 'B-LEGAL_CONCEPT', 'I-LEGAL_CONCEPT', 'I-LEGAL_CONCEPT', 'I-LEGAL_CONCEPT', 'I-LEGAL_CONCEPT', 'I-LEGAL_CONCEPT', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


## 4. Xây dựng label space

Gom toàn bộ label từ cả 3 split để đảm bảo không bị miss entity type ở val/test. `O` luôn được đặt ở index 0.

In [5]:
label_counter = Counter()
for rows in (train_rows, val_rows, test_rows):
    for r in rows:
        label_counter.update(r["ner_tags"])

all_labels = sorted(label_counter.keys())
if "O" in all_labels:
    all_labels.remove("O")
LABEL_LIST = ["O"] + all_labels

label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(LABEL_LIST)

print(f"NUM_LABELS = {NUM_LABELS}")
print("Top label counts:")
for l, c in label_counter.most_common(15):
    print(f"  {l:<30} {c}")

# map string labels -> int
def encode_tags(rows):
    for r in rows:
        r["ner_tags"] = [label2id[t] for t in r["ner_tags"]]
    return rows

train_rows = encode_tags(train_rows)
val_rows   = encode_tags(val_rows)
test_rows  = encode_tags(test_rows)

raw_ds = DatasetDict({
    "train":      Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
    "test":       Dataset.from_list(test_rows),
})
raw_ds

NUM_LABELS = 125
Top label counts:
  O                              253473
  I-SANCTION                     61873
  I-LEGAL_ACTOR                  47292
  I-VIOLATION                    37099
  I-LAW                          34403
  I-LEGAL_ACTION                 28856
  I-CONDITION                    26846
  I-SYSTEM                       22589
  I-ATTACK                       15738
  I-STANDARD                     15105
  I-TIME                         11171
  I-DEPLOYMENT                   10275
  B-OBLIGATION                   9516
  I-SECURITY_MEASURE             8317
  I-INCIDENT                     8110


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 6403
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1279
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1288
    })
})

## 5. Tokenizer + căn chỉnh label cho subword

Chiến lược chuẩn cho NER:
- First subtoken của mỗi word → giữ nguyên label.
- Các subtoken còn lại → gán `-100` (CrossEntropyLoss sẽ bỏ qua).
- Special tokens `[CLS]`, `[SEP]`, padding → `-100`.

→ Loss chỉ tính trên 1 đại diện cho mỗi word, đồng thời metric seqeval trả về đúng label theo word-level sau khi decode.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        aligned = []
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev_word:
                aligned.append(word_labels[wid])
            else:
                aligned.append(-100)  # subword sau của cùng 1 word
            prev_word = wid
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = raw_ds.map(
    tokenize_and_align,
    batched=True,
    remove_columns=raw_ds["train"].column_names,
    desc="Tokenizing + aligning labels",
)
tokenized_ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Tokenizing + aligning labels:   0%|          | 0/6403 [00:00<?, ? examples/s]

Tokenizing + aligning labels:   0%|          | 0/1279 [00:00<?, ? examples/s]

Tokenizing + aligning labels:   0%|          | 0/1288 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6403
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1279
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1288
    })
})

## 6. Model — load base + thay head cho label mới

Vì head gốc của `NlpHUST/ner-vietnamese-electra-base` được train trên bộ tag khác, ta dùng `ignore_mismatched_sizes=True` để HF tự khởi tạo lại classifier layer với `NUM_LABELS` của mình, trong khi backbone vẫn giữ trọng số Electra đã train.

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
print(model.config.model_type, "| params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

model.safetensors:   0%|          | 0.00/532M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ElectraForTokenClassification LOAD REPORT from: NlpHUST/ner-vietnamese-electra-base
Key                             | Status     |                                                                                       
--------------------------------+------------+---------------------------------------------------------------------------------------
electra.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.weight               | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9, 768]) vs model:torch.Size([125, 768])
classifier.bias                 | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9]) vs model:torch.Size([125])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


electra | params: 133.162877 M


## 7. Layer-wise Learning Rate Decay (LLRD)

Các layer càng gần input càng học feature tổng quát (cần LR nhỏ), các layer gần head cần LR lớn hơn. LLRD nhân LR theo hệ số `LLRD_DECAY^(N - layer_idx)`.

Head (classifier) dùng `CLASSIFIER_LR` riêng — cao hơn vì phải học từ đầu.

In [8]:
def build_llrd_optimizer(model, base_lr, classifier_lr, weight_decay, decay=0.9):
    no_decay = ("bias", "LayerNorm.weight")

    # Electra: model.electra.embeddings + model.electra.encoder.layer.{0..N-1}
    backbone = getattr(model, "electra", None) or getattr(model, "base_model")
    num_layers = len(backbone.encoder.layer)

    groups = []

    # embeddings = layer 0
    lr_emb = base_lr * (decay ** (num_layers + 1))
    emb_params = list(backbone.embeddings.named_parameters())
    groups.append({
        "params": [p for n, p in emb_params if not any(nd in n for nd in no_decay)],
        "lr": lr_emb, "weight_decay": weight_decay,
    })
    groups.append({
        "params": [p for n, p in emb_params if any(nd in n for nd in no_decay)],
        "lr": lr_emb, "weight_decay": 0.0,
    })

    # encoder layers
    for i, layer in enumerate(backbone.encoder.layer):
        lr_i = base_lr * (decay ** (num_layers - i))
        params = list(layer.named_parameters())
        groups.append({
            "params": [p for n, p in params if not any(nd in n for nd in no_decay)],
            "lr": lr_i, "weight_decay": weight_decay,
        })
        groups.append({
            "params": [p for n, p in params if any(nd in n for nd in no_decay)],
            "lr": lr_i, "weight_decay": 0.0,
        })

    # optional modules: embeddings_project (Electra base đôi khi có)
    if hasattr(backbone, "embeddings_project"):
        proj_params = list(backbone.embeddings_project.named_parameters())
        groups.append({
            "params": [p for n, p in proj_params if not any(nd in n for nd in no_decay)],
            "lr": lr_emb, "weight_decay": weight_decay,
        })
        groups.append({
            "params": [p for n, p in proj_params if any(nd in n for nd in no_decay)],
            "lr": lr_emb, "weight_decay": 0.0,
        })

    # classifier head — LR cao hơn
    head_params = []
    for name, p in model.named_parameters():
        if not name.startswith(("electra", "base_model")):
            head_params.append((name, p))
    groups.append({
        "params": [p for n, p in head_params if not any(nd in n for nd in no_decay)],
        "lr": classifier_lr, "weight_decay": weight_decay,
    })
    groups.append({
        "params": [p for n, p in head_params if any(nd in n for nd in no_decay)],
        "lr": classifier_lr, "weight_decay": 0.0,
    })

    # bỏ group rỗng
    groups = [g for g in groups if len(g["params"]) > 0]
    return torch.optim.AdamW(groups, lr=base_lr)

print("LLRD optimizer builder sẵn sàng.")

LLRD optimizer builder sẵn sàng.


## 8. Metric — seqeval entity-level

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for pred_seq, lab_seq in zip(preds, labels):
        cur_t, cur_p = [], []
        for p_i, l_i in zip(pred_seq, lab_seq):
            if l_i == -100:
                continue
            cur_t.append(id2label[int(l_i)])
            cur_p.append(id2label[int(p_i)])
        true_labels.append(cur_t)
        true_preds.append(cur_p)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall":    recall_score(true_labels, true_preds),
        "f1":        f1_score(true_labels, true_preds),
    }

## 9. Training

In [10]:
import inspect

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding="longest",
    label_pad_token_id=-100,
)

# Xây dựng kwargs theo argument được hỗ trợ bởi phiên bản transformers hiện tại
# (tương thích cả bản cũ và bản v5+).
_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
_tr_params = set(inspect.signature(Trainer.__init__).parameters.keys())

ta_kwargs = dict(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=BATCH_SIZE_EVAL,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",
    max_grad_norm=MAX_GRAD_NORM,
    label_smoothing_factor=LABEL_SMOOTHING,

    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,
    report_to="none",

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    seed=SEED,
)

# --- evaluation strategy: tên arg đã đổi qua nhiều version ---
if "eval_strategy" in _ta_params:
    ta_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in _ta_params:
    ta_kwargs["evaluation_strategy"] = "epoch"

# --- warmup: v5 khuyến nghị warmup_steps; bản cũ dùng warmup_ratio ---
# Ước lượng tổng số update step để suy ra warmup_steps tương đương WARMUP_RATIO.
_steps_per_epoch = max(
    1,
    len(tokenized_ds["train"]) // (BATCH_SIZE_TRAIN * GRAD_ACCUM),
)
_total_steps = _steps_per_epoch * NUM_EPOCHS
_warmup_steps = int(_total_steps * WARMUP_RATIO)
if "warmup_steps" in _ta_params:
    ta_kwargs["warmup_steps"] = _warmup_steps
elif "warmup_ratio" in _ta_params:
    ta_kwargs["warmup_ratio"] = WARMUP_RATIO
print(f"warmup_steps ≈ {_warmup_steps} / {_total_steps} total steps")

# --- precision / misc chỉ có ở bản mới ---
if "data_seed" in _ta_params:
    ta_kwargs["data_seed"] = SEED
if "bf16" in _ta_params:
    ta_kwargs["bf16"] = USE_BF16
if "fp16" in _ta_params:
    ta_kwargs["fp16"] = USE_FP16
if "tf32" in _ta_params and HAS_CUDA:
    ta_kwargs["tf32"] = True

# logging_dir đã deprecate ở v5 — ưu tiên env var, fallback kwargs cho bản cũ.
os.environ["TENSORBOARD_LOGGING_DIR"] = str(LOG_DIR)
if "logging_dir" in _ta_params:
    # bản cũ vẫn chấp nhận, không warning ở các bản <5.0
    import transformers as _tf
    _major = int(_tf.__version__.split(".")[0])
    if _major < 5:
        ta_kwargs["logging_dir"] = str(LOG_DIR)

training_args = TrainingArguments(**ta_kwargs)

optimizer = build_llrd_optimizer(
    model,
    base_lr=BASE_LR,
    classifier_lr=CLASSIFIER_LR,
    weight_decay=WEIGHT_DECAY,
    decay=LLRD_DECAY,
)

# v5 đổi `tokenizer` → `processing_class`. Auto-detect.
tr_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),   # scheduler tự build theo warmup
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PAT)],
)
if "processing_class" in _tr_params:
    tr_kwargs["processing_class"] = tokenizer
elif "tokenizer" in _tr_params:
    tr_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**tr_kwargs)

print("=== Bắt đầu train ===")
train_result = trainer.train()
print(train_result.metrics)

warmup_steps ≈ 300 / 3000 total steps
=== Bắt đầu train ===


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,3.627763,3.514860,0.000000,0.000000,0.000000
2,2.673621,2.226996,0.205702,0.235264,0.219492
3,1.640676,1.283220,0.597242,0.714231,0.650518
4,1.189001,1.010418,0.832732,0.885437,0.858276
5,1.031608,0.923978,0.900740,0.934024,0.917080
6,0.939858,0.885474,0.937070,0.957678,0.947262
7,0.889845,0.865330,0.956768,0.970592,0.963631
8,0.872619,0.852466,0.966250,0.977369,0.971777
9,0.854024,0.844500,0.973254,0.981716,0.977467
10,0.843122,0.838509,0.980557,0.986575,0.983556


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

{'train_runtime': 242.6868, 'train_samples_per_second': 791.514, 'train_steps_per_second': 12.485, 'total_flos': 2.344407913800225e+16, 'train_loss': 1.106367756993295, 'epoch': 28.0}


## 10. Đánh giá trên tập test

In [11]:
val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation:", val_metrics)

test_metrics = trainer.evaluate(tokenized_ds["test"])
print("Test:", test_metrics)

# classification_report chi tiết theo từng entity type
preds_output = trainer.predict(tokenized_ds["test"])
pred_ids = np.argmax(preds_output.predictions, axis=-1)
label_ids = preds_output.label_ids

y_true, y_pred = [], []
for p_seq, l_seq in zip(pred_ids, label_ids):
    t, p = [], []
    for p_i, l_i in zip(p_seq, l_seq):
        if l_i == -100:
            continue
        t.append(id2label[int(l_i)])
        p.append(id2label[int(p_i)])
    y_true.append(t)
    y_pred.append(p)

print("\n=== seqeval classification report (TEST) ===")
print(classification_report(y_true, y_pred, digits=4))

Validation: {'eval_loss': 0.820510745048523, 'eval_precision': 0.996676891615542, 'eval_recall': 0.9970591995908452, 'eval_f1': 0.996868008948546, 'eval_runtime': 0.6102, 'eval_samples_per_second': 2096.115, 'eval_steps_per_second': 16.389, 'epoch': 28.0}
Test: {'eval_loss': 0.8181455731391907, 'eval_precision': 0.9945320447609359, 'eval_recall': 0.9968136630130002, 'eval_f1': 0.995671546785487, 'eval_runtime': 0.5902, 'eval_samples_per_second': 2182.467, 'eval_steps_per_second': 18.639, 'epoch': 28.0}

=== seqeval classification report (TEST) ===
                      precision    recall  f1-score   support

         APPLICATION     1.0000    1.0000    1.0000         3
             ARTICLE     1.0000    1.0000    1.0000       101
              ATTACK     0.9943    1.0000    0.9972       175
               AUDIT     0.8750    1.0000    0.9333         7
       AUTHORIZATION     1.0000    1.0000    1.0000         3
           BANDWIDTH     1.0000    1.0000    1.0000         1
         CE

## 11. Lưu model + label mapping

In [12]:
OUTPUT_DIR = PROJECT_ROOT / "/content/models" / "ner-legal-vi-electra"

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

with open(OUTPUT_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

print("Đã lưu model tại:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu model tại: /content/models/ner-legal-vi-electra
['label_mapping.json', 'training_args.bin', 'model.safetensors', 'tokenizer.json', 'config.json', 'tokenizer_config.json']


In [14]:
import shutil
import os

# Create a zip file of the output directory
output_zip_path = "models.zip"
shutil.make_archive(output_zip_path.replace(".zip", ""), 'zip', OUTPUT_DIR)
print(f"Created {output_zip_path} which you can now download.")

# Optionally, you can list the current directory to see the created zip file
# !ls -lh

Created models.zip which you can now download.


## 12. Demo inference

Gom các subword về word-level và merge liên tiếp `B-X` / `I-X` thành entity cuối cùng.

In [15]:
from transformers import pipeline

device_id = 0 if torch.cuda.is_available() else -1
ner_pipe = pipeline(
    "token-classification",
    model=str(OUTPUT_DIR),
    tokenizer=str(OUTPUT_DIR),
    aggregation_strategy="simple",
    device=device_id,
)

examples = [
"Theo khoản 2 Điều 15 của Luật An ninh mạng, doanh nghiệp viễn thông cung cấp dịch vụ internet phải thực hiện lưu trữ dữ liệu cá nhân của người sử dụng tại Việt Nam trong thời gian tối thiểu 24 tháng, trừ trường hợp pháp luật có quy định khác, và phải cung cấp thông tin này cho Bộ Công an khi có yêu cầu phục vụ công tác điều tra."
]
for text in examples:
    print("\n→", text)
    for ent in ner_pipe(text):
        print(f"  {ent['entity_group']:<20} '{ent['word']}'  (score={ent['score']:.3f})")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


→ Theo khoản 2 Điều 15 của Luật An ninh mạng, doanh nghiệp viễn thông cung cấp dịch vụ internet phải thực hiện lưu trữ dữ liệu cá nhân của người sử dụng tại Việt Nam trong thời gian tối thiểu 24 tháng, trừ trường hợp pháp luật có quy định khác, và phải cung cấp thông tin này cho Bộ Công an khi có yêu cầu phục vụ công tác điều tra.
  CLAUSE               'khoản 2'  (score=0.954)
  ARTICLE              'Điều 15'  (score=0.957)
  LAW                  'của'  (score=0.395)
  LAW                  'Luật An ninh mạng'  (score=0.932)
  TELECOM_OPERATOR     'doanh nghiệp viễn thông'  (score=0.758)
  TELECOM_OPERATOR     'internet'  (score=0.749)
  OBLIGATION           'phải'  (score=0.933)
  LEGAL_ACTION         'thực hiện lưu trữ dữ liệu cá nhân'  (score=0.873)
  LEGAL_ACTION         'người'  (score=0.634)
  TIME                 'tối thiểu 24 tháng'  (score=0.947)
  OBLIGATION           'phải'  (score=0.940)
  LEGAL_ACTION         'cung cấp thông tin'  (score=0.863)
  LEGAL_ACTOR          'Bộ 

## 13. (Tuỳ chọn) Export nhanh — tích hợp vào `backend/graph_rag/ner_extractor.py`

Sau khi hài lòng với kết quả, bạn có thể load model trong code Python:

```python
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

MODEL_DIR = "models/ner-legal-vi-electra"
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
ner = pipeline("token-classification", model=mdl, tokenizer=tok, aggregation_strategy="simple")
```